# IDS - Network Intrusion Detection

Analise Exploratoria e Tratamento de Dados


## 1. Setup - Importacoes


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

BASE = '../datas'
# encoding latin-1: os labels deste CSV tem caracteres fora do UTF-8 (ex.: 'Web Attack \x96 Brute Force')
web = pd.read_csv(f'{BASE}/raw/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', encoding='latin-1')


## 2. Exploracao dos Dados


In [ ]:
print('===== Web Attacks Dataset =====')
print('Shape:', web.shape)
print('\nTipos:\n', web.dtypes)


## 3. Limpeza e Tratamento


In [ ]:
web.columns = web.columns.str.strip()
web['Label'] = web['Label'].str.strip()

### 3.1 Valores Infinitos


In [ ]:
# Converter infinitos em NaN
web.replace([np.inf, -np.inf], np.nan, inplace=True)


### 3.2 Valores Ausentes


In [ ]:
print('\nValores ausentes:')
print(web.isna().sum())

# Remover NaN em Flow Bytes/s
web.dropna(subset=['Flow Bytes/s', 'Flow Packets/s'], inplace=True)



### 3.3 Duplicatas


In [ ]:
print('Linhas duplicadas:', web.duplicated().sum())

web.drop_duplicates(inplace=True)


### 3.4 Codificacao de Labels


In [ ]:

le = LabelEncoder()
web['Label'] = le.fit_transform(web['Label'])


## 4. Preparacao para Modelagem


In [ ]:
X = web.drop('Label', axis=1)
y = web['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 5. Modelagem


### 5.1 Decision Tree


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print('Decision Tree Accuracy:', accuracy_score(y_test, dt.predict(X_test)))


### 5.2 Random Forest


In [ ]:
rf = RandomForestClassifier(n_estimators=15, random_state=42)
rf.fit(X_train, y_train)
print('Random Forest Accuracy:', accuracy_score(y_test, rf.predict(X_test)))


### 5.3 XGBoost


In [ ]:
xgb = XGBClassifier(n_estimators=10, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
print('XGBoost Accuracy:', accuracy_score(y_test, xgb.predict(X_test)))


## Analise Detalhada com Metricas


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Decision Tree predictions
dt_pred = dt.predict(X_test)

print('='*50)
print('DECISION TREE - METRICAS')
print('='*50)
print(f'Acuracia: {accuracy_score(y_test, dt_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred))


In [ ]:
cm_dt = confusion_matrix(y_test, dt_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Matriz de Confusao - Decision Tree')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predito')
plt.show()


In [ ]:
feat_dt = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_dt, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Features - Decision Tree')
plt.tight_layout()
plt.show()


In [ ]:
rf_pred = rf.predict(X_test)

print('='*50)
print('RANDOM FOREST - METRICAS')
print('='*50)
print(f'Acuracia: {accuracy_score(y_test, rf_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred))


In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', cbar=True)
plt.title('Matriz de Confusao - Random Forest')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predito')
plt.show()


In [ ]:
feat_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_rf, x='Importance', y='Feature', palette='rocket')
plt.title('Top 15 Features - Random Forest')
plt.tight_layout()
plt.show()


In [ ]:
xgb_pred = xgb.predict(X_test)

print('='*50)
print('XGBOOST - METRICAS')
print('='*50)
print(f'Acuracia: {accuracy_score(y_test, xgb_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, xgb_pred))


In [ ]:
cm_xgb = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Reds', cbar=True)
plt.title('Matriz de Confusao - XGBoost')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predito')
plt.show()


In [ ]:
feat_xgb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_xgb, x='Importance', y='Feature', palette='mako')
plt.title('Top 15 Features - XGBoost')
plt.tight_layout()
plt.show()


## SVM - Support Vector Machine


In [ ]:
# Treinar SVM Linear (mais rápido)
svm = LinearSVC(random_state=42, C=1.0, max_iter=10000)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

print('='*50)
print('SVM Linear - METRICAS')
print('='*50)
print(f'Acuracia: {accuracy_score(y_test, svm_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, svm_pred))

In [ ]:
cm_svm = confusion_matrix(y_test, svm_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='YlOrRd', cbar=True)
plt.title('Matriz de Confusao - SVM')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predito')
plt.show()


In [ ]:
# Para SVM Linear, usar coeficientes absolutos para importância de features
feat_svm = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(svm.coef_).mean(axis=0)  # media entre classes (funciona para binario e multiclasse)
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_svm, x='Importance', y='Feature', palette='coolwarm')
plt.title('Top 15 Features - SVM Linear (Coefficient Importance)')
plt.tight_layout()
plt.show()

## MLP (Multilayer Perceptron)

In [ ]:
# Treinar MLP
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42, early_stopping=True)
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)

print('='*50)
print('MLP - METRICAS')
print('='*50)
print(f'Acuracia: {accuracy_score(y_test, mlp_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, mlp_pred))

In [ ]:
# Matriz de Confusão - MLP
cm_mlp = confusion_matrix(y_test, mlp_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusão - MLP')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()

In [ ]:
# Calcular importância de features com permutation importance - MLP
perm_importance_mlp = permutation_importance(mlp, X_test, y_test, n_repeats=10, random_state=42)

feat_mlp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance_mlp.importances_mean
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_mlp, x='Importance', y='Feature', palette='coolwarm')
plt.title('Top 15 Features - MLP (Permutation Importance)')
plt.tight_layout()
plt.show()

## Comparacao Final - Todos os Modelos


In [ ]:
# Comparar todos os modelos
resultados = pd.DataFrame({
    'Modelo': ['Decision Tree', 'Random Forest', 'XGBoost', 'SVM', 'MLP'],
    'Acuracia': [
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, svm_pred),
        accuracy_score(y_test, mlp_pred)
    ]
})

print('\n' + '='*50)
print('COMPARACAO FINAL - TODOS OS MODELOS')
print('='*50)
print(resultados.to_string(index=False))
print(f'\nMelhor Modelo: {resultados.loc[resultados["Acuracia"].idxmax(), "Modelo"]}')
print('='*50)


In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)